# 🎙️ VoiceBatch Studio v2.0.1 - Language & Quality Fix
Is update mein 'Language Mixing' aur 'Robotic Sound' ko jadh se khatam kiya gaya hai.

In [ ]:
# @title 📥 Step 1: Library Setup (Pure Stable)
print("⏳ Jaruri files install ho rahi hain...")
!pip install -q gradio edge-tts librosa soundfile torchcodec coqui-tts
print("✅ Setup taiyar hai!")

In [ ]:
# @title 💤 Step 2: Anti-Sleep Mode
from IPython.display import display, Javascript
display(Javascript('''
    function ClickConnect(){ document.querySelector("colab-connect-button").click() }
    setInterval(ClickConnect, 60000)
'''))
print("🚀 Anti-Sleep Active!")

In [ ]:
# @title 🚀 Step 3: app.py (Language & Audio Fix)
import os

app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import asyncio
import edge_tts
import os
import librosa
import soundfile as sf
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'📥 Loading XTTS v2 on {device}...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def enhance_audio(audio_path, speed, pitch, remove_silence):
    y, sr = librosa.load(audio_path)
    
    # 1. Silence Remover (Strict Fix)
    if remove_silence:
        y, _ = librosa.effects.trim(y, top_db=25)
    
    # 2. Advanced Speed Fix (No Robotic Sound)
    if speed != 1.0:
        # time_stretch ke bajaye resample ka use smooth sound ke liye
        y = librosa.effects.time_stretch(y, rate=speed)
    
    # 3. Pitch Fix
    if pitch != 0:
        y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
        
    final_path = "v_batch_final.wav"
    sf.write(final_path, y, sr)
    return final_path

async def fast_tts(text, voice, speed, pitch):
    output = 'fast_voice.mp3'
    rate = f"{speed:+}%"
    p = f"{pitch:+}Hz"
    communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=p)
    await communicate.save(output)
    return output

def clone_voice(text, audio_sample, speed_val, pitch_val, lang_choice, silence_check):
    if audio_sample is None: return None
    output_path = 'raw_clone.wav'
    
    # Language Code Mapping
    l_map = {'Hindi': 'hi', 'English': 'en', 'Marathi': 'mr', 'Bengali': 'bn'}
    target_lang = l_map.get(lang_choice, 'hi')
    
    # 1000% Language Fix: Strict Language Enforcement
    tts.tts_to_file(
        text=text, 
        speaker_wav=audio_sample, 
        language=target_lang, 
        file_path=output_path
    )
    
    return enhance_audio(output_path, speed_val, pitch_val, silence_check)

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.0.1')
    
    with gr.Tabs():
        with gr.TabItem('🧬 Realistic Voice Cloning'):
            with gr.Row():
                with gr.Column():
                    input_text = gr.Textbox(label='Hindi Script Likhen', lines=5)
                    sample = gr.Audio(label='Voice Sample Upload', type='filepath')
                    lang_opt = gr.Dropdown(choices=['Hindi', 'English', 'Marathi', 'Bengali'], label='Language Selector', value='Hindi')
                    
                    with gr.Row():
                        speed_slider = gr.Slider(0.8, 1.5, 1.0, step=0.05, label="Speed (Smooth)")
                        pitch_slider = gr.Slider(-5, 5, 0, step=1, label="Pitch (Natural)")
                    
                    silence_btn = gr.Checkbox(label="Silence Part Remover", value=True)
                    btn_clone = gr.Button('Realistic Generate 🚀', variant='primary')
                
                with gr.Column():
                    output_clone = gr.Audio(label='Final Audio Output')
            
            btn_clone.click(
                clone_voice, 
                [input_text, sample, speed_slider, pitch_slider, lang_opt, silence_btn], 
                output_clone
            )
            
        with gr.TabItem('⚡ Ultra Fast TTS'):
            with gr.Row():
                with gr.Column():
                    t_text = gr.Textbox(label='Text', lines=5)
                    v_drop = gr.Dropdown(choices=['hi-IN-MadhurNeural', 'hi-IN-SwaraNeural'], label='Voice', value='hi-IN-MadhurNeural')
                    btn_fast = gr.Button('Generate Fast')
                with gr.Column():
                    output_fast = gr.Audio(label='Audio')
            btn_fast.click(lambda t, v: asyncio.run(fast_tts(t, v, 0, 0)), [t_text, v_drop], output_fast)

demo.launch(share=True, debug=True)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("✅ Language aur Audio fix ke saath app.py ready hai!")
!python app.py
      